# Protein Design · 空白模板

只验证目录、运行身份与反馈；靶标和计算步骤尚未填写，默认状态为 **NOT_RUN**。


## 开始前

从 DesignProject 目录启动。输入、科学参数或源码变化应新建运行；恢复须指定原目录，且严格身份一致。


## 启动设置

填写 RUN_LABEL。当前模板不安装依赖、不下载模型、不运行 GPU。


In [ ]:
# 【可修改设置】启动与环境设置：包路径、输出位置、恢复开关、不加载科学引擎。
# SR56_FEEDBACK_BEGIN
import sys as _feedback_sys, os as _feedback_os, importlib.util as _feedback_importlib
from pathlib import Path as _feedback_Path
import hashlib as _feedback_hashlib

def _feedback_safe(operation):
    try:
        return operation()
    except Exception as error:
        print('[SR56 feedback disabled operation] ' + type(error).__name__ + ': ' + str(error), file=_feedback_sys.stderr)

class _FeedbackNull:
    def event(self, *args, **kwargs): pass
    def configure(self, *args, **kwargs): pass
    def finish(self, *args, **kwargs): pass

_standalone_previous_feedback = globals().get("_feedback")

# 直接运行设置：通常留空即可；从任意工作目录运行时填写完整包目录。
# 包的绝对路径；留空时从当前目录/父目录定位。移动包或从其他工作目录启动时可填写。
STANDALONE_PACKAGE_ROOT = ""
# 留空：每次生成新运行目录。恢复：填写原目录，并设下面的 RESUME=True。
# 本次独立运行根目录；留空自动新建。恢复时填写原目录；禁止用新版本接管旧版本断点。
STANDALONE_OUTPUT_ROOT = ""
# 实验短名；留空为未命名实验。仅影响文件夹名称，不改变科学参数。
RUN_LABEL = ""
# False 创建新运行；True 恢复上面指定、身份匹配的运行；不会复制历史候选到新运行。
STANDALONE_RESUME = False
# 固定资源身份，由发布程序生成；不要手动修改或用它绕过资源完整性检查。
RESOURCE_BUNDLE_SHA256 = 'd655131c0b6c3107d58d3031a05231336532186ffa2f7db0fd67b661f136e74f'

def _prepare_resources(package):
    import json, zipfile, stat, tempfile, shutil, fcntl, io
    archive=package/'resources.zip'
    if archive.is_symlink() or not archive.is_file():
        raise RuntimeError('缺少发布根 resources.zip；请保留完整运行包')
    if archive.stat().st_size>64*1024**2: raise RuntimeError('资源 ZIP 超出大小上限')
    raw=archive.read_bytes()
    if _feedback_hashlib.sha256(raw).hexdigest()!=RESOURCE_BUNDLE_SHA256:
        raise RuntimeError('resources.zip 散列不符；拒绝加载其他版本或损坏资源')
    cache=package/'.resources'
    if cache.is_symlink(): raise RuntimeError('资源缓存不能是符号链接')
    cache.mkdir(exist_ok=True)
    dest=cache/RESOURCE_BUNDLE_SHA256
    lockpath=cache/'.prepare.lock'
    if lockpath.is_symlink(): raise RuntimeError('资源缓存锁不能是符号链接')
    with lockpath.open('a+') as lock:
        fcntl.flock(lock.fileno(),fcntl.LOCK_EX)
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            infos=z.infolist(); names=[i.filename for i in infos]
            if len(infos)>128 or len(names)!=len(set(names)) or sum(i.file_size for i in infos)>64*1024**2:
                raise RuntimeError('资源包成员重复或超出大小上限')
            for i in infos:
                p=_feedback_Path(i.filename)
                if (p.is_absolute() or '..' in p.parts or '\\' in i.filename or ':' in i.filename
                        or not p.parts or any(part in ('','.') for part in i.filename.split('/'))
                        or i.is_dir() or stat.S_ISLNK(i.external_attr>>16)
                        or i.flag_bits&1 or i.file_size>16*1024**2):
                    raise RuntimeError('资源包含不安全成员：'+i.filename)
            if 'RESOURCE_MANIFEST.json' not in names:
                raise RuntimeError('资源包缺少清单')
            manifest=json.loads(z.read('RESOURCE_MANIFEST.json'))
            if manifest.get('schema')!='sr56-compact-resources-v1':
                raise RuntimeError('不兼容的资源清单')
            files=manifest['files']; byname={item['path']:item for item in files}
            if len(byname)!=len(files) or set(names)!=(set(byname)|{'RESOURCE_MANIFEST.json'}):
                raise RuntimeError('资源包成员与清单不一致')
            def check(root):
                if root.is_symlink() or not root.is_dir(): raise RuntimeError('资源缓存无效')
                for name,item in byname.items():
                    p=root/name
                    if (any(q.is_symlink() for q in (p,*p.parents) if q!=root.parent)
                            or not p.is_file() or p.stat().st_size!=item['bytes']
                            or _feedback_hashlib.sha256(p.read_bytes()).hexdigest()!=item['sha256']):
                        raise RuntimeError('资源缓存缺失或被修改：'+name)
                if (root/'RESOURCE_MANIFEST.json').read_bytes()!=z.read('RESOURCE_MANIFEST.json'):
                    raise RuntimeError('资源缓存清单被修改')
            if dest.exists() or dest.is_symlink():
                check(dest)
            else:
                temp=_feedback_Path(tempfile.mkdtemp(prefix='.prepare-',dir=cache))
                try:
                    for i in infos:
                        p=temp/i.filename;p.parent.mkdir(parents=True,exist_ok=True)
                        p.write_bytes(z.read(i))
                    check(temp)
                    _feedback_os.replace(temp,dest)
                finally:
                    if temp.exists(): shutil.rmtree(temp)
    return dest


def _feedback_boot():
    global _feedback, _standalone_runtime, _feedback_paths, _feedback_assessed
    if (_standalone_previous_feedback is not None
            and getattr(_standalone_previous_feedback, '_standalone_lock', None)
            and not _standalone_previous_feedback.finished):
        _refusal = RuntimeError('此 kernel 已有运行会话；不要重复启动单元。继续后续单元，或中断并重启 kernel 后显式恢复。')
        _refusal.sr56_setup_rejected = True
        raise _refusal
    if _standalone_previous_feedback is not None and getattr(_standalone_previous_feedback, 'finished', False):
        _standalone_previous_feedback.close()
    cwd = _feedback_Path.cwd()
    candidates = ([_feedback_Path(STANDALONE_PACKAGE_ROOT).expanduser()] if STANDALONE_PACKAGE_ROOT
                  else [cwd, cwd.parent])
    roots = [p.resolve() for p in candidates if (p/'resources.zip').is_file()
             and (p/'DesignProject/DesignProject.ipynb').is_file()]
    roots = list(dict.fromkeys(roots))
    if len(roots) != 1:
        raise RuntimeError('请将工作目录设为本包目录；也可在本单元填写 STANDALONE_PACKAGE_ROOT 绝对路径。')
    package = roots[0]
    resource_root = _prepare_resources(package)
    # Do not import an installed or another cached runtime with the same name.
    runtime_file = resource_root/'standalone_runtime.py'
    runtime_spec = _feedback_importlib.spec_from_file_location('sr56_standalone_runtime', runtime_file)
    _standalone_runtime = _feedback_importlib.module_from_spec(runtime_spec)
    _feedback_sys.modules['sr56_standalone_runtime'] = _standalone_runtime
    exec(compile(runtime_file.read_bytes(), str(runtime_file), 'exec'), _standalone_runtime.__dict__)
    notebook = package/'DesignProject/DesignProject.ipynb'
    editor_path = globals().get('__vsc_ipynb_file__')
    if editor_path and _feedback_Path(editor_path).resolve() != notebook.resolve():
        raise RuntimeError('VS Code 当前 notebook 与所选包入口不一致；请打开包内对应入口。')
    _feedback = _standalone_runtime.start(package=package, notebook=notebook,
        branch='template_project', namespace='design_template_v1', output=STANDALONE_OUTPUT_ROOT,
        resume=STANDALONE_RESUME, resource_root=resource_root, run_label=RUN_LABEL)
    _feedback_paths = {}
    _feedback_assessed = {}
    print('独立运行模式：', 'template_project')
    print('运行 ID：', _feedback.run['run_id'], '执行批次：', _feedback.attempt)
    print('运行目录：', _feedback.root)
    print('无需工作台服务；科学输入和计算步骤尚未配置。')

_feedback_boot()


## 靶标和科学参数

在此填写；留空为未配置，不推断序列、编号或预算。


In [ ]:
TARGET_STRUCTURE = ""
TARGET_SEQUENCE = ""
TARGET_CHAIN = ""
PARAMETERS = {}
CONFIG = {"pipeline_status": "NOT_RUN", "scientific_parameters": PARAMETERS}
TARGET = {"structure": TARGET_STRUCTURE or None, "sequence": TARGET_SEQUENCE or None, "chain": TARGET_CHAIN or None}
_feedback.configure(CONFIG, TARGET, "design_template_v1", _feedback.root)


In [ ]:
RUN_ROOT = _feedback.root
for directory in ("main/rfd3", "main/mpnn", "main/rf3", "refine/rfd3", "refine/mpnn", "refine/rf3",
                  "reports/main", "reports/refine", "rankings/main", "rankings/refine", "svg/main", "svg/refine",
                  "monitor", "checkpoints/main", "checkpoints/refine", "prep", "feedback"):
    (RUN_ROOT / directory).mkdir(parents=True, exist_ok=True)
print("运行目录已准备；科学计算 NOT_RUN:", RUN_ROOT)


## 主阶段

接入实际计算、回执及模型评估后才记录完成数。回执字段和候选身份见 docs/MAINTENANCE.md；可参考 SR56 example。


In [ ]:
# 在此接入 main：RFD3 → MPNN → RF3（尚未实现）
print("main: NOT_RUN")


In [ ]:
# 在此接入 refine、固定 baseline 与全循环选优（尚未实现）
print("refine: NOT_RUN")


## 导出框架验证反馈

下面生成 NOT_RUN 快照和本地页面。接入真实流程后，结束状态必须来自实际执行与验证，不能沿用模板状态冒充完成。


In [ ]:
_feedback.finish('NOT_RUN')
_feedback.close()
print('模板框架验证结束；没有科学计算。')
